# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umerkniazi/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# My lane as an ML task

## Task type: Ranking

My Content Refresh Prioritization lane is a ranking problem.

The goal is to rank pages by their likelihood of needing review, so a content team can inspect the highest-priority pages first.

The output is not a final decision. It is a prioritized list that supports human review.

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target or proxy

The target will be a proxy for pages that may need attention.

A possible target is whether a page is declining based on the observed `trend_direction` value.

The label comes from historical performance data, not from a human decision.

Example:
- `down` trend → declining page (1)
- other trends → not declining (0)

This is a proxy because a declining trend does not mean a page definitely needs updating. It only indicates a pattern worth reviewing.

In [2]:
df["is_declining_target"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

df[["trend_direction", "is_declining_target"]].head()

,trend_direction,is_declining_target
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Success metric

The main success metric is Precision@K.

For example, Precision@50 measures how many of the top 50 recommended pages are actually declining.

This metric matches the business action because a content team usually has limited capacity and wants the highest-value pages at the top of the list.

A higher Precision@50 means fewer wasted reviews.

In [3]:
declining_rate = df["is_declining_target"].mean()

print("Overall declining rate:", round(declining_rate, 3))
print("Pages available for ranking:", len(df))

Overall declining rate: 0.542
Pages available for ranking: 30000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

# Unit of analysis

The unit of analysis is one page.

Each row represents one content page and contains observable signals about that page, such as:
- search impressions
- ranking position
- click-through rate
- content age
- update history

The model would score individual pages and create a prioritized review list.

In [4]:
page_df = df[
    [
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "trend_direction"
    ]
]

page_df.head(10)

,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction
0,3803,10.6,0.76,187,20,down
1,15320,20.3,0.05,445,25,down
2,12581,36.5,0.09,141,20,down
3,11751,6.2,0.49,463,22,stable
4,19140,44.0,0.13,263,14,down
5,3970,8.5,0.03,147,20,down
6,20,7.0,0.00,90,20,down
7,1724,21.2,0.06,445,22,stable
8,32574,46.0,0.09,90,20,down
9,1240,4.9,0.16,257,104,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

# Why ML beats a fixed rule

A fixed rule such as "review every page older than 180 days" only considers one signal.

Content performance depends on multiple factors, including freshness, visibility, ranking position, and engagement.

Machine learning can combine multiple signals and identify patterns that are difficult to write as simple if/else rules.

The goal is not to replace human judgment, but to improve prioritization.

In [5]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

df[features].describe()

,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr
count,30000.00000,30000.000000,30000.000000,30000.00000,30000.000000
mean,256.16780,46.098300,5200.366300,16.34238,0.510733
std,132.70793,42.078709,16838.019547,15.21679,3.279162
min,90.00000,1.000000,1.000000,0.00000,0.000000
25%,132.00000,20.000000,81.000000,6.20000,0.000000
50%,236.00000,20.000000,731.000000,10.80000,0.070000
75%,333.00000,104.000000,3615.250000,22.30000,0.290000
max,564.00000,373.000000,517715.000000,245.00000,100.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.